# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via the Croissant schema URL below. The schema uses Linked Data (`@id`) to refer to all dataset elements.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Display dataset-level metadata
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"License: {dataset.metadata.license}")
print(f"Version: {dataset.metadata.version}")
print(f"Keywords: {', '.join(dataset.metadata.keywords)}")

## 2. Data Overview
Review available record sets and their fields. All references are by their `@id` as defined in the Croissant schema.

In [ ]:
# List the available record sets (`@id` and name field)
print("Available record sets in the dataset:")
record_sets = list(dataset.record_sets)
for rec in record_sets:
    print(f"  - @id: {rec.id}")
    print(f"    name: {rec.name}")
    # List fields (by @id and label)
    print("    fields:")
    for fld in rec.fields:
        print(f"      - @id: {fld.id:50} label: {fld.name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Below, we select the main survey record set, whose `@id` is shown above.

In [ ]:
# Set the record set @id you want to analyze (from above overview)
# For this dataset, the main record set appears to be 'https://api.app.sen.science/frontiers/7160411/794de323-23ec-4496-8bd5-d9c5b848dafe'
main_record_set_id = 'https://api.app.sen.science/frontiers/7160411/794de323-23ec-4496-8bd5-d9c5b848dafe'
record_sets_to_load = [main_record_set_id]

dataframes = {}
for rec_id in record_sets_to_load:
    print(f"Loading records from record set: {rec_id}")
    records = list(dataset.records(record_set=rec_id))
    dataframes[rec_id] = pd.DataFrame(records)

print(f"Fields (DataFrame columns) in record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing operations such as filtering, normalization, and grouping. All field names are referenced by their `@id` as present in the schema/dataframe columns (see above cell output).

In [ ]:
# Select a numeric field for analysis
# For this dataset, GAD-7 total score is commonly included in mental health surveys
# Find the most appropriate field by examining the DataFrame columns
numeric_field_id = 'GAD7_total_score'  # Replace with the exact column name if different

df = dataframes[main_record_set_id]

if numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field if available
    group_field_id = 'gender'  # Replace with exact field name if different
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df)
    else:
        print(f"Grouping field '{group_field_id}' not present in columns.")
else:
    print(f"'{numeric_field_id}' not found in columns. Available columns are:\n{df.columns.tolist()}")

## 5. Visualization
Visualize data distributions or relationships using pandas and matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of GAD7_total_score
if numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], bins=15, kde=True)
    plt.title('Distribution of GAD-7 Total Score')
    plt.xlabel('GAD-7 Total Score')
    plt.ylabel('Count')
    plt.show()

# Boxplot by gender if available
if numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.show()
else:
    print('gender or GAD7_total_score not found in columns for boxplot.')

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library:

- The dataset metadata and structure are accessible via the Croissant schema.
- We listed record sets and fields, then loaded the main survey record set using its `@id`.
- Exploratory data analysis steps included filtering records with high GAD-7 scores, normalizing, and visualizing by gender if available.
- The use of `@id` for all elements ensures reproducible and precise data access.

For further analysis, you are encouraged to consult the schema for additional field IDs and to explore other record sets, fields, and relationships.